# Documentation Assistant
### Data Pipeline

In [1]:
import io
import zipfile
import traceback
from dataclasses import dataclass

import requests

import frontmatter
from typing import List, Dict, Any, Iterable, Callable
from minsearch import Index

import json

from openai import OpenAI

In [2]:
@dataclass
class RawRepositoryFile:
    filename: str
    content: str


class GithubRepositoryDataReader:
    """
    Downloads and parses markdown and code files from a GitHub repository.
    """

    def __init__(self,
                repo_owner: str,
                repo_name: str,
                allowed_extensions: Iterable[str] | None = None,
                filename_filter: Callable[[str], bool] | None = None
        ):
        """
        Initialize the GitHub repository data reader.
        
        Args:
            repo_owner: The owner/organization of the GitHub repository
            repo_name: The name of the GitHub repository
            allowed_extensions: Optional set of file extensions to include
                    (e.g., {"md", "py"}). If not provided, all file types are included
            filename_filter: Optional callable to filter files by their path
        """
        prefix = "https://codeload.github.com"
        self.url = (
            f"{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main"
        )

        if allowed_extensions is not None:
            self.allowed_extensions = {ext.lower() for ext in allowed_extensions}

        if filename_filter is None:
            self.filename_filter = lambda filepath: True
        else:
            self.filename_filter = filename_filter

    def read(self) -> list[RawRepositoryFile]:
        """
        Download and extract files from the GitHub repository.
        
        Returns:
            List of RawRepositoryFile objects for each processed file
            
        Raises:
            Exception: If the repository download fails
        """
        resp = requests.get(self.url)
        if resp.status_code != 200:
            raise Exception(f"Failed to download repository: {resp.status_code}")

        zf = zipfile.ZipFile(io.BytesIO(resp.content))
        repository_data = self._extract_files(zf)
        zf.close()

        return repository_data

    def _extract_files(self, zf: zipfile.ZipFile) -> list[RawRepositoryFile]:
        """
        Extract and process files from the zip archive.
        
        Args:
            zf: ZipFile object containing the repository data

        Returns:
            List of RawRepositoryFile objects for each processed file
        """
        data = []

        for file_info in zf.infolist():
            filepath = self._normalize_filepath(file_info.filename)

            if self._should_skip_file(filepath):
                continue

            try:
                with zf.open(file_info) as f_in:
                    content = f_in.read().decode("utf-8", errors="ignore")
                    if content is not None:
                        content = content.strip()

                    file = RawRepositoryFile(
                        filename=filepath,
                        content=content
                    )
                    data.append(file)

            except Exception as e:
                print(f"Error processing {file_info.filename}: {e}")
                traceback.print_exc()
                continue

        return data

    def _should_skip_file(self, filepath: str) -> bool:
        """
        Determine whether a file should be skipped during processing.
        
        Args:
            filepath: The file path to check
            
        Returns:
            True if the file should be skipped, False otherwise
        """
        filepath = filepath.lower()

        # directory
        if filepath.endswith("/"):
            return True

        # hidden file
        filename = filepath.split("/")[-1]
        if filename.startswith("."):
            return True

        if self.allowed_extensions:
            ext = self._get_extension(filepath)
            if ext not in self.allowed_extensions:
                return True

        if not self.filename_filter(filepath):
            return True

        return False

    def _get_extension(self, filepath: str) -> str:
        """
        Extract the file extension from a filepath.
        
        Args:
            filepath: The file path to extract extension from
            
        Returns:
            The file extension (without dot) or empty string if no extension
        """
        filename = filepath.lower().split("/")[-1]
        if "." in filename:
            return filename.rsplit(".", maxsplit=1)[-1]
        else:
            return ""

    def _normalize_filepath(self, filepath: str) -> str:
        """
        Removes the top-level directory from the file path inside the zip archive.
        'repo-main/path/to/file.py' -> 'path/to/file.py'
        
        Args:
            filepath: The original filepath from the zip archive
            
        Returns:
            The normalized filepath with top-level directory removed
        """
        parts = filepath.split("/", maxsplit=1)
        if len(parts) > 1:
            return parts[1]
        else:
            return parts[0]

In [3]:
def read_github_data() -> List[RawRepositoryFile]:
    repo_owner = "evidentlyai"
    repo_name = "docs"

    allowed_extensions = {"md", "mdx"}

    reader = GithubRepositoryDataReader(
        repo_owner,
        repo_name,
        allowed_extensions=allowed_extensions,
    )
    
    return reader.read()

In [4]:
data_raw = read_github_data()

In [7]:
data_raw[:3]

[RawRepositoryFile(filename='api-reference/endpoint/create.mdx', content="---\ntitle: 'Create Plant'\nopenapi: 'POST /plants'\n---"),
 RawRepositoryFile(filename='api-reference/endpoint/delete.mdx', content="---\ntitle: 'Delete Plant'\nopenapi: 'DELETE /plants/{id}'\n---"),
 RawRepositoryFile(filename='api-reference/endpoint/get.mdx', content="---\ntitle: 'Get Plants'\nopenapi: 'GET /plants'\n---")]

In [ ]:
def parse_data(data_raw: List[RawRepositoryFile]) -> List[Dict[str, Any]]:
    """
    Format each instance in an array of Repo file objects
    """
    data_parsed = []
    for f in data_raw:
        # format the file content
        post = frontmatter.loads(f.content)
        data = post.to_dict()

        # keep track of file name
        data['filename'] = f.filename
        
        data_parsed.append(data)

    return data_parsed

In [25]:
parsed_data = parse_data(data_raw)

In [26]:
parsed_data[40]

{'title': 'Evidently and GitHub actions',
 'description': 'Testing LLM outputs as part of the CI/CD flow.',
 'content': 'You can use Evidently together with GitHub Actions to automatically test the outputs of your LLM agent or application - as part of every code push or pull request.\n\n## How the integration work:\n\n- You define a test dataset of inputs (e.g. test prompts with or without reference answers). You can store it as a file, or save the dataset at Evidently Cloud callable by Dataset ID.\n- Run your LLM system or agent against those inputs inside CI.\n- Evidently automatically evaluates the outputs using the user-specified config (which defines the Evidently descriptors, tests and Report composition), including methods like:\n  - LLM judges (e.g., tone, helpfulness, correctness)\n  - Custom Python functions\n  - Dataset-level metrics like classification quality\n- If any test fails, the CI job fails.\n- You get a detailed test report with pass/fail status and metrics.\n\n![]

### Chunking

In [ ]:
def sliding_window(
        seq: Iterable[Any],
        size: int,
        step: int
    ) -> List[Dict[str, Any]]:
    """
    Create overlapping chunks from a sequence using a sliding window approach.

    Args:
        seq: The input sequence (string or list) to be chunked.
        size (int): The size of each chunk/window.
        step (int): The step size between consecutive windows.

    Returns:
        list: A list of dictionaries, each containing:
            - 'start': The starting position of the chunk in the original sequence
            - 'content': The chunk content

    Raises:
        ValueError: If size or step are not positive integers.

    Example:
        >>> sliding_window("hello world", size=5, step=3)
        [{'start': 0, 'content': 'hello'}, {'start': 3, 'content': 'lo wo'}]
    """
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        batch = seq[i:i+size] # get the next sequence of <size> characters
        result.append({'start': i, 'content': batch})
        if i + size > n:
            break

    return result


def chunk_documents(
        documents: Iterable[Dict[str, str]],
        size: int = 2000,
        step: int = 1000,
        content_field_name: str = 'content'
) -> List[Dict[str, str]]:
    """
    Split a collection of documents into smaller chunks using sliding windows.

    Takes documents and breaks their content into overlapping chunks while preserving
    all other document metadata (filename, etc.) in each chunk.

    Args:
        documents: An iterable of document dictionaries. Each document must have a content field.
        size (int, optional): The maximum size of each chunk. Defaults to 2000.
        step (int, optional): The step size between chunks. Defaults to 1000.
        content_field_name (str, optional): The name of the field containing document content.
                                          Defaults to 'content'.

    Returns:
        list: A list of chunk dictionaries. Each chunk contains:
            - All original document fields except the content field
            - 'start': Starting position of the chunk in original content
            - 'content': The chunk content

    Example:
        >>> documents = [{'content': 'long text...', 'filename': 'doc.txt'}]
        >>> chunks = chunk_documents(documents, size=100, step=50)
        >>> # Or with custom content field:
        >>> documents = [{'text': 'long text...', 'filename': 'doc.txt'}]
        >>> chunks = chunk_documents(documents, content_field_name='text')
    """
    results = []

    for doc in documents:
        doc_copy = doc.copy()
        doc_content = doc_copy.pop(content_field_name) # extract and remove the document content
        chunks = sliding_window(doc_content, size=size, step=step) # chunk the context
        
        # add back the chunked content
        for chunk in chunks:
            chunk.update(doc_copy)
        results.extend(chunks)

    return results

In [28]:
def index_documents(documents, chunk: bool = False, chunking_params=None) -> Index:
    """
    Create a searchable index from a collection of documents.

    Args:
        documents: A collection of document dictionaries, each containing at least
                  'content' and 'filename' fields.
        chunk (bool, optional): Whether to chunk documents before indexing.
                               Defaults to False.
        chunking_params (dict, optional): Parameters for document chunking.
                                        Defaults to {'size': 2000, 'step': 1000}.
                                        Only used when chunk=True.

    Returns:
        Index: A fitted minsearch Index object ready for searching.

    Example:
        >>> docs = [{'content': 'Hello world', 'filename': 'doc1.txt'}]
        >>> index = index_documents(docs)
        >>> results = index.search('hello')
    """
    if chunk:
        if chunking_params is None:
            chunking_params = {'size': 2000, 'step': 1000}
        documents = chunk_documents(documents, **chunking_params)

    index = Index(
        text_fields=["content", "filename"],
    )

    index.fit(documents)
    return index

In [ ]:
def index_faq_data():
    repo_owner = "DataTalksClub"
    repo_name = "faq"

    # fetch and format the data in the repo's files
    data_raw = read_github_data(repo_owner, repo_name)
    documents = parse_data(data_raw)

    index = index_documents(
        documents,
        chunk=True,
        chunking_params={"size": 2000, "step": 1000},
    )

    return index

In [31]:
index = index_documents(
    parsed_data,
    chunk=True,
    chunking_params={"size": 2000, "step": 1000},
)

### RAG

In [ ]:
# create search engine based on chunked documents
def search(query):
    return index.search(
        query=query,
        num_results=15
    )

In [35]:
instructions = """
    You're an assistant that helps with the documentation.
    Answer the QUESTION based on the CONTEXT from the search engine of our documentation.

    Use only the facts from the CONTEXT when answering the QUESTION.

    When answering the question, provide the reference to the file with the source.
    Use the filename field for that. The repo url is: https://github.com/evidentlyai/docs/
    Include code examples when relevant. 
    If the question is discussed in multiple documents, cite all of them.

    Don't use markdown or any formatting in the output.
""".strip()

prompt_template = """
    <QUESTION>
    {question}
    </QUESTION>

    <CONTEXT>
    {context}
    </CONTEXT>
""".strip()


def build_prompt(question, search_results):
    context = json.dumps(search_results)

    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    
    return prompt

In [37]:
openai_client = OpenAI()

In [38]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [39]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    response = llm(prompt)
    return response

In [40]:
question = "How can I build an eval report with llm as a judge?"

In [41]:
result = rag(question)
print(result)

To build an evaluation report using a Large Language Model (LLM) as a judge, you can follow these structured steps:

### Step 1: Installation and Setup
First, ensure that you have the necessary library installed. You'll need `evidently` for generating reports.

```python
pip install evidently[llm]
```

After installation, import the required modules:

```python
import pandas as pd
import os
from evidently import Dataset, Report
from evidently.presets import TextEvals
```

Next, set your OpenAI API key:

```python
os.environ["OPENAI_API_KEY"] = "YOUR_KEY"
```

### Step 2: Create Your Evaluation Dataset
Build an evaluation dataset. This typically includes:

- **Questions**: The inputs you want to evaluate.
- **Target Responses**: Approved (correct) responses to compare against.
- **New Responses**: Responses generated by the LLM app.
- **Manual Labels**: Labels indicating whether the generated responses are correct.

Example of creating a dataset:

```python
data = {
    "question": ["Wh